# Creating a dendrogram displaying tool similarity

Based on the tools' heatmaps (not predictions) we calculate how similar the tools' results are and display those findings in a dendrogram

### Steps 

- Load in all heatmap data for the HGSOC experiment
- Reduce heatmap to comparing one modality against another (instead of all vs all)
- Fill empty / NaNs
- Make sure sample ordering is alphabetical and the same across tools
- Make each heatmap matrix into a vector
- Scale each heatmap so that 0 indicates a pair of samples is identical and 1 indicates they are very different
- Create array of observation vectors (i.e. matrix of tools x scaled heatmap values)
- Use scipy's linkage to perform hierarchical clustering on the array of observation vectors
- Use scipy's dendrogram for visualization

## Load & prep data

In [1]:
import numpy as np

from analysis_shared_functions import load_heatmap_data

In [2]:
DATA_PATH = "/Users/lathouwe/Documents/HGSOC/hgsoc-reconstruction/benchmarking/data/output_data/vcf_update"

### CrosscheckFingerprints

Linkage disequilibrium method. 

Sample similarity interpretation: negative values are non-matches, positive values are matches, and values close to zero are inconclusive.

There is no universal upper or lower bound.

We're going to map the highest value to zero (match) and the lowest value to one (non-match). Inconclusives will be lost unless no value was found at all.

In [40]:
heatmap = load_heatmap_data(
    DATA_PATH=DATA_PATH,
    pseudobulk=False,
    tool="Peddy",
    dataset="hgsoc-new",
    ncells="null",
    rd=0,
    mod1="bulk_chunk_ribo",
    mod2="bulk_diss_polyA",
)

Processing tool Peddy, dataset hgsoc-new, read depth 0...


In [23]:
conpair_heatmap = load_heatmap_data(
    DATA_PATH=DATA_PATH,
    pseudobulk=False,
    tool="Conpair",
    dataset="hgsoc-new",
    ncells="null",
    rd=0,
    mod1="bulk_chunk_ribo",
    mod2="bulk_diss_polyA",
)

Processing tool Conpair, dataset hgsoc-new, read depth 0...


In [24]:
conpair_heatmap.columns = conpair_heatmap.columns.str.replace("_bulk_chunk_ribo", "", 1).str.replace("_bulk_diss_polyA", "", 1)
conpair_heatmap.index = conpair_heatmap.index.str.replace("_bulk_chunk_ribo", "", 1).str.replace("_bulk_diss_polyA", "", 1)


In [25]:
# Which sample is missing in the CrosscheckFingerprints heatmap
missing_samples = set(conpair_heatmap.columns) - set(heatmap.columns)
missing_samples

set()

In [26]:
print(heatmap.min().min(), heatmap.max().max())

-0.9467 40.0


In [42]:
# Keep only bulk_chunk_ribo samples in the rows and bulk_diss_polyA in the columns
heatmap = heatmap.loc[
    heatmap.index.str.contains("bulk_chunk_ribo"),
    heatmap.columns.str.contains("bulk_diss_polyA"),
]

In [43]:
heatmap

,HGSOC-2466_bulk_diss_polyA,HGSOC-2477_bulk_diss_polyA,HGSOC-2455_bulk_diss_polyA,HGSOC-2186_bulk_diss_polyA,HGSOC-2416_bulk_diss_polyA,HGSOC-2483_bulk_diss_polyA,HGSOC-2526_bulk_diss_polyA,HGSOC-2514_bulk_diss_polyA,HGSOC-2507_bulk_diss_polyA,HGSOC-2246_bulk_diss_polyA,...,HGSOC-2313_bulk_diss_polyA,HGSOC-2094_bulk_diss_polyA,HGSOC-2407_bulk_diss_polyA,HGSOC-2202_bulk_diss_polyA,HGSOC-2423_bulk_diss_polyA,HGSOC-2129_bulk_diss_polyA,HGSOC-2401_bulk_diss_polyA,HGSOC-2230_bulk_diss_polyA,HGSOC-2238_bulk_diss_polyA,HGSOC-2408_bulk_diss_polyA
HGSOC-2268_bulk_chunk_ribo,1.2340,0.9306,1.0640,1.0910,1.9820,1.0730,1.4000,1.1100,38.0,1.7710,...,3.1740,0.9050,0.9525,0.8660,0.6261,1.1020,0.8490,1.2070,1.6970,0.8563
HGSOC-2238_bulk_chunk_ribo,0.7868,0.4937,0.5654,0.5920,1.5410,0.6478,0.8128,0.7555,16.0,1.2010,...,1.9900,0.4377,0.4194,0.3966,0.8114,0.6683,0.3920,1.4380,1.1440,0.4240
HGSOC-2309_bulk_chunk_ribo,0.8045,0.6008,0.6421,0.7817,1.6260,0.6835,0.8798,0.8773,26.0,1.2290,...,-0.7763,0.5840,0.5517,0.5090,0.9716,0.7119,0.5142,1.5050,1.2030,0.5090
HGSOC-2507_bulk_chunk_ribo,0.9806,0.5929,0.6887,0.9055,1.0090,0.8766,1.1770,-0.8186,18.0,1.4430,...,2.4380,0.5920,0.6139,0.5257,0.9162,0.8403,0.5687,1.4820,1.4640,0.6648
HGSOC-2423_bulk_chunk_ribo,1.0240,0.8153,0.7952,0.9736,1.6680,0.8805,1.1100,0.8781,24.0,1.4590,...,2.6480,0.7299,0.7764,0.6847,0.4899,0.9296,0.6847,0.9588,1.5670,0.6947
HGSOC-2455_bulk_chunk_ribo,0.7383,0.4047,0.5517,0.6057,1.5640,0.6131,0.8326,0.7026,16.0,1.1420,...,2.1550,0.4313,0.3847,0.3511,0.5949,0.7052,0.3236,1.2100,1.0900,0.4220
HGSOC-2408_bulk_chunk_ribo,0.9628,0.5896,0.7335,0.8396,-0.8262,0.8380,1.1200,0.4225,14.0,1.2920,...,2.4930,0.5973,0.5998,0.5220,0.9150,0.8514,0.5138,1.5090,1.4970,0.6225
HGSOC-2202_bulk_chunk_ribo,0.5784,0.4398,0.4521,0.5249,1.4150,0.5321,0.6895,0.6978,20.0,0.9492,...,1.7370,0.3968,0.3771,-0.8587,0.7850,0.5909,0.3219,1.3230,1.1190,0.3120
HGSOC-2514_bulk_chunk_ribo,0.5606,0.2584,0.3823,0.4080,1.3000,0.4717,0.6438,0.5982,3.0,0.9054,...,1.8220,0.2156,0.2684,0.1996,0.6997,0.4840,0.2221,1.2590,0.9365,0.2829
HGSOC-2221_bulk_chunk_ribo,1.2600,0.8706,1.0140,1.0760,2.0230,1.1180,1.4090,0.9977,40.0,1.8670,...,3.1090,0.8593,0.8797,0.8400,0.6030,1.1040,0.8184,-0.8049,1.7310,0.8286


In [29]:
print(heatmap.min().min(), heatmap.max().max())

-0.9467 40.0


In [30]:
# Are there any missing values?
print(heatmap.isna().sum().sum())

0


In [31]:
crosscheck_max = heatmap.max().max()
crosscheck_min = heatmap.min().min()

In [32]:
scaled_heatmap = heatmap.apply(lambda x: (crosscheck_max - x) / (crosscheck_max - crosscheck_min))

In [33]:
# Verify that the scaled crosscheck heatmap has the expected range
print(scaled_heatmap.min().min(), scaled_heatmap.max().max())

0.0 1.0


In [34]:
# Convert df to numpy array
scaled_heatmap_numpy = scaled_heatmap.to_numpy()
crosscheck_vector = scaled_heatmap_numpy.reshape(-1)

In [35]:
crosscheck_vector

array([0.94674296, 0.95415259, 0.9508947 , ..., 0.94036882, 0.94750004,
       0.96342807], shape=(1152,))

In [36]:

def create_scaled_heatmap_vector(tool, dataset, mod1, mod2):
    heatmap = load_heatmap_data(
        DATA_PATH=DATA_PATH,
        pseudobulk=False,
        tool=tool,
        dataset=dataset,
        ncells="null",
        rd=0,
        mod1=mod1,
        mod2=mod2,
    )
    # Keep only mod1 samples in the rows and mod2 in the columns
    heatmap = heatmap.loc[
        heatmap.index.str.contains(mod1),
        heatmap.columns.str.contains(mod2),
    ]
    if tool == "Conpair":
        heatmap.columns = heatmap.columns.str.replace(f"_{mod1}", "", 1).str.replace(f"_{mod2}", "", 1)
        heatmap.index = heatmap.index.str.replace(f"_{mod1}", "", 1).str.replace(f"_{mod2}", "", 1)

    # Remove sample HGSOC-2507_bulk_diss_polyA (if present)
    if "HGSOC-2507_bulk_diss_polyA" in heatmap.index:
        heatmap = heatmap.drop(index="HGSOC-2507_bulk_diss_polyA")
    if "HGSOC-2507_bulk_diss_polyA" in heatmap.columns:
        heatmap = heatmap.drop(columns="HGSOC-2507_bulk_diss_polyA")

    heatmap_max = heatmap.max().max()
    heatmap_min = heatmap.min().min()
    if tool in ["Peddy", "TimeAttackGenComp", "Vireo"]:
        scaled_heatmap = heatmap.apply(lambda x: (x - heatmap_min) / (heatmap_max - heatmap_min))
    else:
        scaled_heatmap = heatmap.apply(lambda x: (heatmap_max - x) / (heatmap_max - heatmap_min))
    vector = scaled_heatmap.to_numpy().reshape(-1)

    return vector


In [63]:
crosscheck_vector = create_scaled_heatmap_vector("CrosscheckFingerprints", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
crosscheck_vector

Processing tool CrosscheckFingerprints, dataset hgsoc-new, read depth 0...


array([0.39446557, 0.68785467, 0.75130679, ..., 0.68063098, 0.69482012,
       0.68445563], shape=(1116,))

In [64]:
conpair_vector = create_scaled_heatmap_vector("Conpair", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
conpair_vector

Processing tool Conpair, dataset hgsoc-new, read depth 0...


array([0.03318113, 0.84687976, 0.86605784, ..., 0.93409437, 0.83150685,
       0.9391172 ], shape=(1116,))

In [20]:
hysys_vector = create_scaled_heatmap_vector("HYSYS", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
hysys_vector

Processing tool HYSYS, dataset hgsoc-new, read depth 0...


array([0.41750842, 0.82154882, 0.80808081, ..., 0.84511785, 0.81818182,
       0.86868687], shape=(1116,))

In [18]:
ngscheckmate_vector = create_scaled_heatmap_vector("NGSCheckmate", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
ngscheckmate_vector

Processing tool NGSCheckmate, dataset hgsoc-new, read depth 0...


array([0.2038835 , 0.66019417, 0.22330097, ..., 0.68932039, 0.44660194,
       0.51456311], shape=(1116,))

In [7]:
ntsm_vector = create_scaled_heatmap_vector("ntsm", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
ntsm_vector

Processing tool ntsm, dataset hgsoc-new, read depth 0...


array([0.27851397, 1.        , 1.        , ..., 1.        , 1.        ,
       1.        ], shape=(1116,))

In [19]:
omicsprint_vector = create_scaled_heatmap_vector("OmicsPrint", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
omicsprint_vector

Processing tool OmicsPrint, dataset hgsoc-new, read depth 0...


array([0.03707591, 0.6410793 , 0.62800036, ..., 0.87745908, 0.73938715,
              nan], shape=(1116,))

In [37]:
peddy_vector = create_scaled_heatmap_vector("Peddy", "hgsoc-new", "bulk_chunk_ribo", "bulk_diss_polyA")
peddy_vector

Processing tool Peddy, dataset hgsoc-new, read depth 0...


array([0.5292062 , 0.45557794, 0.48795108, ..., 0.59254496, 0.52168321,
       0.36340913], shape=(1116,))